## 안다르 무신사 리뷰 데이터 전처리

In [68]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


### 전처리 시작

In [69]:
# andar 리뷰 데이터 로드
df = pd.read_csv('./check_data/andar_musinsa_review_final_s2024.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6376 entries, 0 to 6375
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   collect_date       6376 non-null   str    
 1   platform           6376 non-null   str    
 2   review_id          6376 non-null   int64  
 3   product_id         6376 non-null   int64  
 4   product_name       6376 non-null   str    
 5   review_date        6376 non-null   str    
 6   year               6376 non-null   int64  
 7   month              6376 non-null   int64  
 8   content            6376 non-null   str    
 9   rating             6376 non-null   int64  
 10  helpful_count      6376 non-null   int64  
 11  has_image          6376 non-null   int64  
 12  purchase_option    6370 non-null   str    
 13  user_size          0 non-null      float64
 14  user_height        4750 non-null   float64
 15  user_weight        4741 non-null   float64
 16  user_height_group  4750 non-null   

In [70]:
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,user_size,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21,무신사,62257472,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-07-09,2024,7,색깔이 그레이에 초록 한스푼 넣은 느낌이에요 디자인이 깔끔해서 너무 맘에 듭니다 두...,5,0,1,F · 포그 그레이[EJFMT-01LGY],NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
1,2026-04-21,무신사,61262848,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-06-25,2024,6,뭔가 바닥 매트 같은 소재여서 로고나 디자인은 예쁘나 조금 아쉽.. 뻐덩뻐덩 한 느...,5,0,1,F · 포그 그레이[EJFMT-01LGY],NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
2,2026-04-21,무신사,59417236,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-05-15,2024,5,안다르 매트 너무 부드럽고 색감도 이뻐요 재질 좋은거쓰시는듯,5,0,1,F · 포레스트 그린[EJFMT-01GRN],NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
3,2026-04-21,무신사,59408897,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-05-15,2024,5,"아침에 스트레칭 하거나 아이가 닌텐도 링핏 할때도 사용하는데 완전 좋아요,,,",5,1,1,F · 포그 그레이[EJFMT-01LGY],NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
4,2026-04-21,무신사,58192969,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-04-19,2024,4,아기가 따라 운동 하고 폭싱 폭싱 하니까 점프 점프 하고 놀아요 ㅎㅎ,5,22,1,F · 파스텔 민트[EJFMT-01MNT],NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...


In [71]:
# user_size 컬럼 삭제
df.drop(columns=['user_size'], inplace=True)

In [72]:
opt = df['purchase_option']

# 컬럼 초기화
df['purchase_option_color'] = pd.NA
df['purchase_option_size']  = pd.NA

# ─────────────────────────────────────────────
# Step 0A : 무신사 세트 포맷  "컬러[SKU]_사이즈 · 컬러[SKU]_사이즈"
#           ]_ 패턴으로 사이즈가 SKU 뒤에 붙는 방식 (세트/번들 구매)
# ─────────────────────────────────────────────
mask_0a = (
    opt.str.contains('·', regex=False, na=False) &
    opt.str.contains(r'\]_', regex=True, na=False)
)

df.loc[mask_0a, 'purchase_option_color'] = (
    opt[mask_0a].apply(
        lambda s: ','.join(m.strip() for m in re.findall(r'([^·\[\]_]+?)\s*\[', s) if m.strip())
    )
)
df.loc[mask_0a, 'purchase_option_size'] = (
    opt[mask_0a].str.extract(r'\]_([^\s·\[,_]+)')[0]
)
print(f"Step 0A (세트 ]_사이즈 포맷)    : {mask_0a.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 0B : 무신사 역순 포맷  "컬러[SKU] · 사이즈"
#           [SKU]가 · 앞에, · 뒤에는 단순 사이즈만
# ─────────────────────────────────────────────
mask_0b = (
    df['purchase_option_color'].isna() &
    opt.str.contains(r'\[.*?\]\s*·', regex=True, na=False) &
    ~opt.str.contains(r'·.*\[', regex=True, na=False)
)

df.loc[mask_0b, 'purchase_option_color'] = (
    opt[mask_0b].str.extract(r'^(.*?)\s*\[')[0].str.strip()
)
df.loc[mask_0b, 'purchase_option_size'] = (
    opt[mask_0b].str.extract(r'·\s*(.+)$')[0].str.strip()
)
print(f"Step 0B (역순 컬러[SKU]·사이즈) : {mask_0b.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 0C : 무신사 기본 포맷  "사이즈 · 컬러[SKU]"
#           중점(·) 구분자 방식
# ─────────────────────────────────────────────
mask_0c = df['purchase_option_color'].isna() & opt.str.contains('·', regex=False, na=False)

df.loc[mask_0c, 'purchase_option_size'] = (
    opt[mask_0c].str.extract(r'^([^·]+)\s*·')[0].str.strip()
)
df.loc[mask_0c, 'purchase_option_color'] = (
    opt[mask_0c].str.extract(r'·\s*(.*?)\s*\[')[0].str.strip()
)
print(f"Step 0C (기본 사이즈·컬러[SKU]) : {mask_0c.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 1 : 레이블 기반  (색상= / 사이즈= 명시된 행)
#          세트 상품은 색상이 여러 개 → findall로 전부 추출 후 , 연결
# ─────────────────────────────────────────────
mask1 = opt.str.contains(r'색상\s*\d*\)?\s*=', na=False, regex=True)

df.loc[mask1, 'purchase_option_color'] = (
    opt[mask1]
    .str.findall(r'색상\s*\d*\)?\s*=\s*(?:\[.*?\])?\s*([^,]+)')
    .apply(lambda lst: ','.join(m.strip() for m in lst) if lst else pd.NA)
)
df.loc[mask1, 'purchase_option_size'] = (
    opt[mask1]
    .str.findall(r'사이즈\s*\d*\)?\s*=\s*([^,]+)')
    .apply(lambda lst: ','.join(m.strip() for m in lst) if lst else pd.NA)
)
print(f"Step 1 (레이블 기반)          : {mask1.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 2a : [SKU] 컬러, 사이즈  (purchase_option이 [로 시작)
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask2a   = mask_rem & opt.str.match(r'^\[', na=False)

cleaned_2a = opt[mask2a].str.replace(r'\[.*?\]\s*', '', regex=True).str.strip()
df.loc[mask2a, 'purchase_option_color'] = cleaned_2a.str.split(',', n=1).str[0].str.strip()
df.loc[mask2a, 'purchase_option_size']  = cleaned_2a.str.split(',', n=1).str[1].str.strip()
print(f"Step 2a ([SKU] 컬러, 사이즈)   : {mask2a.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 2b : 사이즈, [SKU] 컬러  (중간에 , [ 패턴)
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask2b   = mask_rem & opt.str.contains(r',\s*\[', na=False, regex=True)

df.loc[mask2b, 'purchase_option_size'] = (
    opt[mask2b].str.extract(r'^(.*?),\s*\[')[0].str.strip()
)
df.loc[mask2b, 'purchase_option_color'] = (
    opt[mask2b].str.replace(r'^.*?,\s*\[.*?\]\s*', '', regex=True).str.strip()
)
print(f"Step 2b (사이즈, [SKU] 컬러)   : {mask2b.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 3 : 앞 [ 누락된 SKU 패턴  →  사이즈, SKU] 컬러
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask3    = mask_rem & opt.str.contains(r',\s*[A-Z0-9\-]+\]', na=False, regex=True)

df.loc[mask3, 'purchase_option_size'] = (
    opt[mask3].str.extract(r'^(.*?),\s*[A-Z0-9\-]+\]')[0].str.strip()
)
df.loc[mask3, 'purchase_option_color'] = (
    opt[mask3].str.replace(r'^.*?,\s*[A-Z0-9\-]+\]\s*', '', regex=True).str.strip()
)
print(f"Step 3 (앞 [ 누락 SKU)        : {mask3.sum():,}건 처리")

# ─────────────────────────────────────────────
# 미처리 최종 확인
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
print(f"\n최종 미처리: {mask_rem.sum():,}건")
if mask_rem.sum() > 0:
    print(opt[mask_rem].value_counts().head(10))

Step 0A (세트 ]_사이즈 포맷)    : 1,421건 처리
Step 0B (역순 컬러[SKU]·사이즈) : 96건 처리
Step 0C (기본 사이즈·컬러[SKU]) : 4,853건 처리
Step 1 (레이블 기반)          : 0건 처리
Step 2a ([SKU] 컬러, 사이즈)   : 0건 처리
Step 2b (사이즈, [SKU] 컬러)   : 0건 처리
Step 3 (앞 [ 누락 SKU)        : 0건 처리

최종 미처리: 0건


In [73]:
# ─────────────────────────────────────────────
# 후처리: 컬러 정제 (토큰별 개별 정제, 중복 유지)
# ─────────────────────────────────────────────

# 8.2부(기장) + 8.2(표기 혼용) 모두 처리하기 위해 8.2부 → 8.2 순으로 배치
fix_words  = r'숏|롱|미니|스탠다드핏|스탠다드|9부|8\.2부|8\.2|4\.5부|3\.5부|퍼포먼스|후크형|노후크형|슬림핏|오버핏|볼륨업|HOLE|SOLID|맨즈'
_prefix = re.compile(rf'^({fix_words})\s*')
_suffix = re.compile(rf'\s*({fix_words})$')
_sku1   = re.compile(r'\[.*?\]')
_sku2   = re.compile(r'^[A-Z0-9\-]+\]\s*')
_paren  = re.compile(r'\(.*?\)')
_space  = re.compile(r'\s+')

def _clean_token(c):
    c = _sku1.sub('', c)
    c = _sku2.sub('', c)
    c = _paren.sub('', c)
    c = _prefix.sub('', c)
    c = _suffix.sub('', c)
    return _space.sub('', c).strip()

def clean_multicolor(s):
    if pd.isna(s) or s == '':
        return pd.NA
    tokens = [_clean_token(t) for t in s.replace('&', ',').split(',')]
    tokens = [t for t in tokens if t]   # 빈 문자열만 제거, 중복은 유지
    return ','.join(tokens) if tokens else pd.NA

df['purchase_option_color'] = df['purchase_option_color'].apply(clean_multicolor)

print(f"purchase_option_color non-null: {df['purchase_option_color'].notna().sum():,} ({df['purchase_option_color'].notna().mean():.1%})")
print(f"고유 컬러 수: {df['purchase_option_color'].nunique():,}개")
print()
print(df['purchase_option_color'].value_counts().head(20))

purchase_option_color non-null: 6,370 (99.9%)
고유 컬러 수: 485개

purchase_option_color
블랙              1841
블랙,블랙            293
화이트              253
블랙,이클립스네이비       175
블라썸핑크            173
페털라일락            143
블랙,딥나이트          141
클라우드바이올렛         101
파스텔민트             96
포그그레이             95
로즈핑크              95
앤트러사이트그레이         94
포레스트그린            91
블랙,앤트러사이트그레이      58
딥나이트,블랙           50
다크애쉬              48
프레쉬크림             47
모카베이지             46
딥나이트,이클립스네이비      43
이클립스네이비,블랙        42
Name: count, dtype: int64


In [74]:
col = df['purchase_option_color'].dropna()

# ① 숫자 포함 → 사이즈 정보 혼입 의심
mask_num = col.str.contains(r'\d', regex=True)
print(f"[숫자 포함] {mask_num.sum():,}건")
print(col[mask_num].value_counts().head(10))

# ② 대문자 영문 포함 → SKU 잔재 의심
mask_upper = col.str.contains(r'[A-Z]', regex=True)
print(f"\n[대문자 영문 포함] {mask_upper.sum():,}건")
print(col[mask_upper].value_counts().head(10))

# ③ 특수문자 잔재 ([, ], =, ,)
mask_special = col.str.contains(r'[\[\]=,]', regex=True)
print(f"\n[특수문자 잔재] {mask_special.sum():,}건")
print(col[mask_special].value_counts().head(10))

# ④ 비정상적으로 긴 컬러명 (10자 초과)
mask_long = col.str.len() > 10
print(f"\n[10자 초과 컬러명] {mask_long.sum():,}건")
print(col[mask_long].value_counts().head(10))

[숫자 포함] 0건
Series([], Name: count, dtype: int64)

[대문자 영문 포함] 0건
Series([], Name: count, dtype: int64)

[특수문자 잔재] 1,421건
purchase_option_color
블랙,블랙           293
블랙,이클립스네이비      175
블랙,딥나이트         141
블랙,앤트러사이트그레이     58
딥나이트,블랙          50
딥나이트,이클립스네이비     43
이클립스네이비,블랙       42
이클립스네이비,딥나이트     27
블랙,셔벗핑크          26
블랙,브라이트화이트       25
Name: count, dtype: int64

[10자 초과 컬러명] 334건
purchase_option_color
블랙,앤트러사이트그레이        58
딥나이트,이클립스네이비        43
이클립스네이비,딥나이트        27
플로리스베이지,플로리스베이지     14
앤트러사이트그레이,블랙        13
플로리스베이지,셔벗핑크        10
위스퍼크림,사하라베이지         7
앤트러사이트그레이,커피빈        7
릴리그린,사하라베이지          5
오트밀베이지,앤트러사이트그레이     5
Name: count, dtype: int64


In [75]:
# 잔여 숫자+부 패턴 추가 정제 (4.5부, 3.5부 등)
_num_boo = re.compile(r'\d+\.?\d*부\s*')

mask_num = df['purchase_option_color'].str.contains(r'\d', na=False, regex=True)
print(f"처리 전: {mask_num.sum():,}건")

df.loc[mask_num, 'purchase_option_color'] = (
    df.loc[mask_num, 'purchase_option_color'].apply(
        lambda s: ','.join(
            t for t in [_num_boo.sub('', tok).strip() for tok in s.split(',')]
            if t
        ) if pd.notna(s) else pd.NA
    )
)

mask_num2 = df['purchase_option_color'].str.contains(r'\d', na=False, regex=True)
print(f"처리 후: {mask_num2.sum():,}건")
if mask_num2.sum() > 0:
    print(df.loc[mask_num2, 'purchase_option_color'].value_counts().head(10))

처리 전: 0건
처리 후: 0건


In [76]:
# 정렬 통일 시 고유 컬러 수 미리 확인 (실제 변경 없음)
sorted_preview = df['purchase_option_color'].dropna().apply(
    lambda s: ','.join(sorted(s.split(','))) if ',' in s else s
)

current  = df['purchase_option_color'].nunique()
after    = sorted_preview.nunique()
single   = df['purchase_option_color'].dropna()
multi_mask = single.str.contains(',', regex=False)

print(f"현재 고유 컬러 수     : {current:,}개")
print(f"정렬 후 예상 고유 수  : {after:,}개  (감소 {current - after:,}개)")
print()
print(f"단품 컬러 행 수       : {(~multi_mask).sum():,}건")
print(f"멀티컬러 행 수        : {multi_mask.sum():,}건")
print(f"  멀티 고유 수(현재)  : {single[multi_mask].nunique():,}개")
print(f"  멀티 고유 수(정렬후): {sorted_preview[multi_mask].nunique():,}개")

현재 고유 컬러 수     : 485개
정렬 후 예상 고유 수  : 428개  (감소 57개)

단품 컬러 행 수       : 4,949건
멀티컬러 행 수        : 1,421건
  멀티 고유 수(현재)  : 182개
  멀티 고유 수(정렬후): 125개


In [77]:
# 멀티컬러 정렬 통일 적용 (블랙,화이트 / 화이트,블랙 → 블랙,화이트)
df['purchase_option_color'] = df['purchase_option_color'].apply(
    lambda s: ','.join(sorted(s.split(','))) if pd.notna(s) and ',' in s else s
)

print(f"고유 컬러 수: {df['purchase_option_color'].nunique():,}개")
print()
print(df['purchase_option_color'].value_counts().head(20))

고유 컬러 수: 428개

purchase_option_color
블랙              1841
블랙,블랙            293
화이트              253
블랙,이클립스네이비       217
딥나이트,블랙          191
블라썸핑크            173
페털라일락            143
클라우드바이올렛         101
파스텔민트             96
포그그레이             95
로즈핑크              95
앤트러사이트그레이         94
포레스트그린            91
블랙,앤트러사이트그레이      71
딥나이트,이클립스네이비      70
다크애쉬              48
프레쉬크림             47
모카베이지             46
브라이트화이트,블랙        36
블랙,셔벗핑크           36
Name: count, dtype: int64


In [78]:
df['purchase_option_color'].value_counts()

purchase_option_color
블랙            1841
블랙,블랙          293
화이트            253
블랙,이클립스네이비     217
딥나이트,블랙        191
              ... 
크림화이트            1
네츄럴베이지           1
더스크라벤더           1
멜란지밀크시럽          1
페르시안블루           1
Name: count, Length: 428, dtype: int64

In [79]:
all_colors = df['purchase_option_color'].dropna().str.split(',').explode()
print(f"총 고유 컬러 수: {all_colors.nunique():,}개")
print()
print(all_colors.value_counts().head())

총 고유 컬러 수: 323개

purchase_option_color
블랙           3213
이클립스네이비       306
딥나이트          300
화이트           256
앤트러사이트그레이     186
Name: count, dtype: int64


In [80]:
# 단일 컬러 기준 이상값 탐지
all_colors = df['purchase_option_color'].dropna().str.split(',').explode().str.strip()
all_colors = all_colors[all_colors != '']

vc = all_colors.value_counts()
print(f"총 고유 컬러 수: {vc.shape[0]:,}개\n")

# ① 숫자 포함
num_mask = all_colors.str.contains(r'\d', regex=True)
print(f"[숫자 포함] {all_colors[num_mask].nunique():,}종  {num_mask.sum():,}건")
print(all_colors[num_mask].value_counts().head(10))

# ② 특수문자 포함 (한글/영문/숫자 외)
sp_mask = all_colors.str.contains(r'[^\w가-힣]', regex=True)
print(f"\n[특수문자 포함] {all_colors[sp_mask].nunique():,}종  {sp_mask.sum():,}건")
print(all_colors[sp_mask].value_counts().head(10))

# ③ 1글자 이하
short_mask = all_colors.str.len() <= 1
print(f"\n[1글자 이하] {all_colors[short_mask].nunique():,}종  {short_mask.sum():,}건")
print(all_colors[short_mask].value_counts())

# ④ 3건 이하 희소 컬러
print(f"\n[3건 이하 희소 컬러] {(vc <= 3).sum():,}개")
print(vc[vc <= 3].head(20))

총 고유 컬러 수: 323개

[숫자 포함] 0종  0건
Series([], Name: count, dtype: int64)

[특수문자 포함] 0종  0건
Series([], Name: count, dtype: int64)

[1글자 이하] 0종  0건
Series([], Name: count, dtype: int64)

[3건 이하 희소 컬러] 135개
purchase_option_color
카키브라운       3
로즈퍼퓸        3
퍼플헤더        3
피치넥타르       3
파스텔라일락      3
라일락         3
모스카키        3
베티버차콜       3
블루솝         3
에어블루        3
스프루스블루      3
멜란지미스티그린    3
블루밸리        3
투어마린        3
몬베이지        3
다크네이비       3
딥그레이        3
써밋그레이       3
시멘트베이지      3
나이트네이비      3
Name: count, dtype: int64


In [81]:
# # ──────────────────────────────────────────────────
# # purchase_option_size 후처리
# # 실제 사이즈가 아닌 핏/기장 단어를 별도 토큰으로 제거
# # ──────────────────────────────────────────────────
# _drop = re.compile(
#     r'^(숏|롱|미니|스탠다드핏|스탠다드|슬림핏|오버핏|볼륨업|퍼포먼스|후크형|노후크형|맨즈|HOLE|SOLID|9부|8\.2부|4\.5부|3\.5부)$'
# )
# _ws          = re.compile(r'\s+')
# _paren_open  = re.compile(r'\(\s+')
# _paren_close = re.compile(r'\s+\)')

# def clean_size_token(s):
#     s = s.strip()
#     s = _paren_open.sub('(', s)
#     s = _paren_close.sub(')', s)
#     s = _ws.sub(' ', s)
#     return s

# def clean_size(s):
#     if pd.isna(s) or s == '':
#         return pd.NA
#     tokens = [clean_size_token(t) for t in s.split(',')]
#     tokens = [t for t in tokens if t and not _drop.match(t)]
#     return ','.join(tokens) if tokens else pd.NA

# df['purchase_option_size'] = df['purchase_option_size'].apply(clean_size)

# print(f"non-null       : {df['purchase_option_size'].notna().sum():,}건 ({df['purchase_option_size'].notna().mean():.1%})")
# print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개")
# print()

# # 단일 토큰 기준 분석
# all_sz = df['purchase_option_size'].dropna().str.split(',').explode().str.strip()
# all_sz = all_sz[all_sz != '']
# vc = all_sz.value_counts()
# print(f"총 고유 개별 사이즈 수: {vc.shape[0]:,}개\n")
# print("=== 상위 30개 ===")
# print(vc.head(30))
# print()

# # 이상값 탐지
# sp_mask   = all_sz.str.contains(r'[\[\]=]', regex=True)
# long_mask = all_sz.str.len() > 20
# kor_only  = all_sz.str.fullmatch(r'[가-힣]+', na=False)   # 한글만 (숏/롱 잔재 확인)

# print(f"[특수문자 잔재] {sp_mask.sum():,}건")
# if sp_mask.sum(): print(all_sz[sp_mask].value_counts().head(10))

# print(f"[20자 초과]     {long_mask.sum():,}건")
# if long_mask.sum(): print(all_sz[long_mask].value_counts().head(10))

# print(f"[한글만 (핏/기장 잔재 의심)] {kor_only.sum():,}건")
# if kor_only.sum(): print(all_sz[kor_only].value_counts().head(20))

# print(f"\n[3건 이하 희소 사이즈] {(vc <= 3).sum():,}개")
# print(vc[vc <= 3].head(20))

In [82]:
# ──────────────────────────────────────────────────
# purchase_option_size 후처리
# 실제 사이즈가 아닌 핏/기장/패턴 단어를 개별 토큰으로 인식하여 제거
# ──────────────────────────────────────────────────
_drop = re.compile(
    r'^(숏|롱|미니|스탠다드핏|스탠다드|슬림핏|오버핏|볼륨업|퍼포먼스|후크형|노후크형|맨즈|HOLE|SOLID|9부|8\.2부|4\.5부|3\.5부|솔리드|멜란지|와이드|레귤러|내추럴|엑스트라롱)$'
)
_ws          = re.compile(r'\s+')
_paren_open  = re.compile(r'\(\s+')
_paren_close = re.compile(r'\s+\)')

def clean_size_token(s):
    s = str(s).strip()
    s = _paren_open.sub('(', s)
    s = _paren_close.sub(')', s)
    s = _ws.sub(' ', s)
    return s

def clean_size(s):
    if pd.isna(s) or s == '':
        return pd.NA
    
    # 콤마 단위로 분리하여 토큰별 정제
    tokens = [clean_size_token(t) for t in str(s).split(',')]
    
    # 빈 문자열과 _drop 리스트에 정확히 일치하는 토큰 필터링
    tokens = [t for t in tokens if t and not _drop.match(t)]
    
    return ','.join(tokens) if tokens else pd.NA

# 컬럼에 함수 적용
df['purchase_option_size'] = df['purchase_option_size'].apply(clean_size)

# ─────────────────────────────────────────────
# 사이즈 컬럼 검증 로직 결과 출력
# ─────────────────────────────────────────────
print(f"purchase_option_size non-null : {df['purchase_option_size'].notna().sum():,}건 ({df['purchase_option_size'].notna().mean():.1%})")
print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개\n")

print("=== 사이즈 상위 30개 ===")
# 단일 토큰 기준으로 쪼개어(explode) 분석
all_sz = df['purchase_option_size'].dropna().str.split(',').explode().str.strip()
all_sz = all_sz[all_sz != '']
vc = all_sz.value_counts()
print(f"총 고유 개별 사이즈 수: {vc.shape[0]:,}개\n")
print(vc.head(30))
print()

# 이상값 탐지 마스킹
sp_mask   = all_sz.str.contains(r'[\[\]=]', regex=True)
long_mask = all_sz.str.len() > 20
kor_only  = all_sz.str.fullmatch(r'[가-힣]+', na=False)   # 한글만 있는 경우 (잔재 여부)

print(f"[특수문자 잔재] {sp_mask.sum():,}건")
if sp_mask.sum() > 0: 
    print(all_sz[sp_mask].value_counts().head(10))

print(f"\n[20자 초과]     {long_mask.sum():,}건")
if long_mask.sum() > 0: 
    print(all_sz[long_mask].value_counts().head(10))

print(f"\n[한글만 (잔재 의심)] {kor_only.sum():,}건")
if kor_only.sum() > 0: 
    print(all_sz[kor_only].value_counts().head(20))

purchase_option_size non-null : 6,370건 (99.9%)
고유 사이즈 조합 수 : 23개

=== 사이즈 상위 30개 ===
총 고유 개별 사이즈 수: 23개

purchase_option_size
F      1134
4      1025
M       928
L       758
2       641
6       523
S       496
XL      374
8       164
2XL      73
0        57
XS       39
29       34
O        31
28       28
27       19
26       18
3XL      10
34        6
30        4
32        4
31        3
36        1
Name: count, dtype: int64

[특수문자 잔재] 0건

[20자 초과]     0건

[한글만 (잔재 의심)] 0건


In [83]:
# 삭제 전 건수 기록
before_count = len(df)

# 결측치를 빈 문자열로 처리한 후, 길이가 5 초과인 데이터만 남기기 (부정 연산자 ~ 대신 직관적인 조건 사용)
df = df[df['content'].fillna('').str.len() > 5]

# 삭제 후 건수 기록
after_count = len(df)

print(f"삭제 전 데이터 건수: {before_count:,}건")
print(f"삭제 후 데이터 건수: {after_count:,}건")
print(f"삭제된 데이터 건수: {before_count - after_count:,}건")

삭제 전 데이터 건수: 6,376건
삭제 후 데이터 건수: 6,376건
삭제된 데이터 건수: 0건


In [84]:
# 알파벳 O → 숫자 0 표준화 (무신사에서 US 바지 사이즈 0을 O로 혼용 표기)
df['purchase_option_size'] = df['purchase_option_size'].str.replace(r'\bO\b', '0', regex=True)

# 1. 대소문자 무시(?i) 및 다양한 프리사이즈 패턴 정의
pattern_free = r'(?i)(F\s*\(Free\)|one\s*size|\bF\b|\bfree\b)'

# 2. 문자열 내 해당 패턴을 모두 '000'으로 일괄 치환 (콤마 유지)
df['purchase_option_size'] = df['purchase_option_size'].str.replace(pattern_free, '000', regex=True)

# 3. 검증
print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개\n")
print(df['purchase_option_size'].value_counts().head(30))

고유 사이즈 조합 수 : 22개

purchase_option_size
000    1134
4      1025
M       928
L       758
2       641
6       523
S       496
XL      374
8       164
0        88
2XL      73
XS       39
29       34
28       28
27       19
26       18
3XL      10
34        6
30        4
32        4
31        3
36        1
Name: count, dtype: int64


In [85]:
# 여성사이즈 XS(80), S(85), M(90), L(95), XL(100), 2XL(105), 3XL(110) 등 확인 및 WXS(80), WS(85), WM(90), WL(95), WXL(100), W2XL(105), W3XL(110) 등으로 치환

# 1. 여성 사이즈 판별 정규식 (공용 사이즈 보호)
# 알파벳 뒤에 (80), (85) 등 특정 숫자가 붙어있는 경우만 매칭 그룹(\1)으로 캡처
# W가 이미 붙어있다면(W?) 무시하고 알파벳부터 캡처합니다.
pattern_women = r'(?i)\bW?(XS\s*\(\s*80\s*\)|S\s*\(\s*85\s*\)|M\s*\(\s*90\s*\)|L\s*\(\s*95\s*\)|XL\s*\(\s*100\s*\)|2XL\s*\(\s*105\s*\)|3XL\s*\(\s*110\s*\))'

# 2. 파생 컬럼 생성: women_size_yn (1=True / 0=False)
df['women_size_yn'] = df['purchase_option_size'].str.contains(pattern_women, na=False).astype(int)

# 3. 사이즈 치환: 캡처된 그룹(\1) 앞에 강제로 'W'를 붙임
# 예: 'S (85)' -> 'WS (85)', 'WS (85)' -> 'WS (85)' / 단독 'S'나 'M (95)'는 무시됨
df['purchase_option_size'] = (
    df['purchase_option_size']
    .str.replace(pattern_women, r'W\1', regex=True)
)

# ─────────────────────────────────────────────
# 결과 확인
# ─────────────────────────────────────────────
print(f"여성 사이즈(women_size_yn=1) 건수: {df['women_size_yn'].sum():,}건")
print("\n[치환된 사이즈 샘플]")
print(df.loc[df['women_size_yn'] == 1, 'purchase_option_size'].value_counts().head(10))

# ─────────────────────────────────────────────
# 공용/기타 사이즈 보호 확인
# ─────────────────────────────────────────────
print("\n[보호된 공용/기타 사이즈 샘플 (W가 안 붙어야 정상)]")
mask_unisex = (df['women_size_yn'] == 0) & df['purchase_option_size'].str.contains(r'\b(S|M|L)\b', na=False)
print(df.loc[mask_unisex, 'purchase_option_size'].value_counts().head(10))


여성 사이즈(women_size_yn=1) 건수: 0건

[치환된 사이즈 샘플]
Series([], Name: count, dtype: int64)

[보호된 공용/기타 사이즈 샘플 (W가 안 붙어야 정상)]
purchase_option_size
M    928
L    758
S    496
Name: count, dtype: int64


In [86]:
# ========================
# 최종 컬럼 정리
# ========================

# 1) 데이터 형변환
def convert_dtypes(df):
    # 날짜 컬럼
    for col in ["collect_date", "review_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # ID 컬럼
    for col in ["review_id", "product_id"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # 문자열 컬럼
    for col in ["product_name", "content", "purchase_option", "product_url"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # category 컬럼
    for col in ["platform", "purchase_option_color", "purchase_option_size", "user_height_group", "user_weight_group"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # 정수형 컬럼
    for col, dtype in [("year", "Int16"), ("month", "Int8"), ("rating", "Int8"), ("helpful_count", "Int32")]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    # 0/1 플래그 컬럼
    for col in ["has_image", "women_size_yn"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    # 실수형 컬럼
    for col in ["user_height", "user_weight"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df

# 사용
df = convert_dtypes(df)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 6376 entries, 0 to 6375
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   collect_date           6376 non-null   datetime64[us]
 1   platform               6376 non-null   category      
 2   review_id              6376 non-null   string        
 3   product_id             6376 non-null   string        
 4   product_name           6376 non-null   string        
 5   review_date            6376 non-null   datetime64[us]
 6   year                   6376 non-null   Int16         
 7   month                  6376 non-null   Int8          
 8   content                6376 non-null   string        
 9   rating                 6376 non-null   Int8          
 10  helpful_count          6376 non-null   Int32         
 11  has_image              6376 non-null   int8          
 12  purchase_option        6370 non-null   string        
 13  user_height   

In [87]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택 (안전하게)
df = df[
    [col for col in column_order if col in df.columns]
]

# 확인
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21,무신사,62257472,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-07-09,2024,7,색깔이 그레이에 초록 한스푼 넣은 느낌이에요 디자인이 깔끔해서 너무 맘에 듭니다 두...,5,0,1,F · 포그 그레이[EJFMT-01LGY],포그그레이,000,0,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
1,2026-04-21,무신사,61262848,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-06-25,2024,6,뭔가 바닥 매트 같은 소재여서 로고나 디자인은 예쁘나 조금 아쉽.. 뻐덩뻐덩 한 느...,5,0,1,F · 포그 그레이[EJFMT-01LGY],포그그레이,000,0,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
2,2026-04-21,무신사,59417236,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-05-15,2024,5,안다르 매트 너무 부드럽고 색감도 이뻐요 재질 좋은거쓰시는듯,5,0,1,F · 포레스트 그린[EJFMT-01GRN],포레스트그린,000,0,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
3,2026-04-21,무신사,59408897,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-05-15,2024,5,"아침에 스트레칭 하거나 아이가 닌텐도 링핏 할때도 사용하는데 완전 좋아요,,,",5,1,1,F · 포그 그레이[EJFMT-01LGY],포그그레이,000,0,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...
4,2026-04-21,무신사,58192969,3297519,릴렉스 에어소프트 요가매트 (8mm),2024-04-19,2024,4,아기가 따라 운동 하고 폭싱 폭싱 하니까 점프 점프 하고 놀아요 ㅎㅎ,5,22,1,F · 파스텔 민트[EJFMT-01MNT],파스텔민트,000,0,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/3297519#revie...


In [ ]:
# df.to_csv('./final_data/andar_musinsa_review_final_s2024.csv', index=False)